In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import re

# Add the parent directory to Python's module search path
sys.path.append(os.path.abspath('..'))
from src.bpe import BPETokenizer

# 1. Load the trained 32K Tokenizer
tokenizer = BPETokenizer()
vocab_file = '../vocabs/bpe_8000.json'

if os.path.exists(vocab_file):
    print("✅ Loading 8K vocabulary for Fairness & Code-Mixing Audit...")
    tokenizer.load(vocab_file)
else:
    print("❌ Error: 8K vocab not found. Please train it first.")

# 2. Define Linguistic Test Samples
test_cases = {
    "Standard English": "Natural language processing is transforming how we interact with machines.",
    "Hindi (Devanagari)": "प्राकृतिक भाषा प्रसंस्करण मशीनों के साथ हमारे बातचीत करने के तरीके को बदल रहा है।",
    "Code-Mixed (Hinglish)": "NLP models train karna bahut computing intensive process hai aaj kal."
}

results = []

print("\n--- Part 3: Fairness & Disparity Audit ---")
for language, text in test_cases.items():
    eval_data = tokenizer.encode(text)
    
    # Calculate Fragmentation (Tokens per Word)
    words = len(text.split())
    tokens = len(eval_data['tokens'])
    fragmentation = tokens / words if words > 0 else 0
    
    results.append({
        "Language / Group": language,
        "Total Words": words,
        "Total Tokens": tokens,
        "Tokens per Word": round(fragmentation, 2),
        "OOV Rate": round(eval_data['oov_rate'], 4)
    })

df_fairness = pd.DataFrame(results)
display(df_fairness)

# Save for the report
os.makedirs('../reports/', exist_ok=True)
df_fairness.to_csv('../reports/fairness_audit.csv', index=False)


# ==========================================
# 3. Part 4: Script-Aware Tokenization
# ==========================================
print("\n--- Part 4: Script-Aware Pre-tokenization ---")

def script_aware_pretokenize(text):
    """
    Prevents cross-script merging by isolating different character blocks.
    E.g., separates Latin characters (English) from Devanagari (Hindi).
    """
    # Regex to capture distinct script blocks (Latin vs Devanagari/Other)
    # \u0900-\u097F is the Unicode block for Devanagari
    script_blocks = re.findall(r'[a-zA-Z]+|[\u0900-\u097F]+|[^a-zA-Z\u0900-\u097F\s]+', text)
    return script_blocks

mixed_text = "Mujhe artificial intelligence (कृत्रिम बुद्धिमत्ता) pasand hai."
print(f"Original Text: {mixed_text}")
print(f"Script-Aware Blocks: {script_aware_pretokenize(mixed_text)}")



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Loading 8K vocabulary for Fairness & Code-Mixing Audit...

--- Part 3: Fairness & Disparity Audit ---


,Language / Group,Total Words,Total Tokens,Tokens per Word,OOV Rate
0,Standard English,10,15,1.50,0.0000
1,Hindi (Devanagari),15,122,8.13,0.5492
2,Code-Mixed (Hinglish),11,25,2.27,0.0000



--- Part 4: Script-Aware Pre-tokenization ---
Original Text: Mujhe artificial intelligence (कृत्रिम बुद्धिमत्ता) pasand hai.
Script-Aware Blocks: ['Mujhe', 'artificial', 'intelligence', '(', 'कृत्रिम', 'बुद्धिमत्ता', ')', 'pasand', 'hai', '.']
